# 11.11 - Context Construction

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Even perfect retrieval produces a poor answer if the prompt is badly assembled. Context construction formats retrieved chunks with source labels and citations, fits them inside a token budget, and instructs the LLM to answer only from the context.

## 2. Why Does This Matter?

A well-structured grounded prompt determines whether the LLM cites sources, stays on topic, and abstains when information is missing - the difference between a credible RAG answer and a confident guess.

## 3. Prerequisites

Units 11.9 (Retrieval), 11.10 (Reranking).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Assemble retrieved chunks into a grounded prompt with [1][2] citations
- Show the final prompt for inspection
- Count tokens with tiktoken's cl100k_base

## 5. Mental Model

Context construction is writing a briefing document for an expert: label the evidence, order it, and tell the expert exactly what you need.

```text
Retrieved Chunks -> Format with Sources -> Manage Token Budget -> Final Prompt
```


## 6. Setup
We'll construct context from a pretend retrieval result - a list of (source, text) chunks exactly like Chroma returns.

In [1]:
retrieved_chunks = [
    {"source": "faq.md", "page": 2, "text": "Returns are accepted within 30 days of purchase."},
    {"source": "policy.pdf", "page": 5, "text": "The customer pays return shipping unless the item is defective."},
    {"source": "faq.md", "page": 3, "text": "Refunds take 5-7 business days to appear on your statement."},
    {"source": "warranty.txt", "page": 1, "text": "Defective products are replaced free of charge."},
]
print(len(retrieved_chunks), "retrieved chunks")


4 retrieved chunks


## 7. Build the Context with Citations
Each chunk becomes a numbered `[i] Source, page N` block. The prompt then instructs the model to answer ONLY from the context and cite `[i]`.

In [2]:
def build_context(chunks, max_chars=1800):
    parts, used = [], 0
    for i, c in enumerate(chunks, start=1):
        block = f"[{i}] {c['source']}, p.{c['page']}\n{c['text']}"
        if used + len(block) > max_chars:
            break
        parts.append(block)
        used += len(block)
    return "\n\n".join(parts)


def build_prompt(query, context):
    return (
        "You are a helpful assistant. Answer the question using ONLY the provided context. "
        "If the context does not contain enough information, say you don't know. "
        "Cite your sources as [1] or [2].\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )


ctx = build_context(retrieved_chunks)
prompt = build_prompt("What is the return window, and who pays return shipping?", ctx)
print(prompt)


You are a helpful assistant. Answer the question using ONLY the provided context. If the context does not contain enough information, say you don't know. Cite your sources as [1] or [2].

Context:
[1] faq.md, p.2
Returns are accepted within 30 days of purchase.

[2] policy.pdf, p.5
The customer pays return shipping unless the item is defective.

[3] faq.md, p.3
Refunds take 5-7 business days to appear on your statement.

[4] warranty.txt, p.1
Defective products are replaced free of charge.

Question: What is the return window, and who pays return shipping?

Answer:


## 8. Token Counting with tiktoken
`cl100k_base` is a good approximation of the tokenizer used by many practical RAG LLMs. We count the context and the full prompt to check we're inside budget.

In [3]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
ctx_tokens = len(enc.encode(ctx))
prompt_tokens = len(enc.encode(prompt))
print("context tokens :", ctx_tokens)
print("prompt tokens  :", prompt_tokens)
print("within 2000-token budget?", prompt_tokens <= 2000)


context tokens : 87
prompt tokens  : 147
within 2000-token budget? True


## 9. Token Budget Management
If retrieval returns more chunks than fit, we keep the highest-ranked ones and drop the tail. Here we simulate too many chunks and see the builder stop early.

In [4]:
many = retrieved_chunks * 3  # tripled
tight = build_context(many, max_chars=600)
print("kept", tight.count("["), "source blocks under a 600-char budget")
print("used chars:", len(tight))


kept 8 source blocks under a 600-char budget
used chars: 596


## 10. Handling Contradictory Sources
Good prompts tell the model how to behave when sources disagree: note the discrepancy and cite both. This is essential in real corpora.

In [5]:
tricky = [
    {"source": "a.txt", "page": 1, "text": "Returns are allowed within 60 days."},
    {"source": "b.txt", "page": 1, "text": "Returns are allowed within 30 days."},
]
ctx2 = build_context(tricky)
prompt2 = build_prompt("What is the return window?", ctx2) + (
    "\n\nIf the sources conflict, say so explicitly and cite both [1] and [2].")
print("TOKENS:", len(enc.encode(prompt2)))
print(prompt2)


TOKENS: 108
You are a helpful assistant. Answer the question using ONLY the provided context. If the context does not contain enough information, say you don't know. Cite your sources as [1] or [2].

Context:
[1] a.txt, p.1
Returns are allowed within 60 days.

[2] b.txt, p.1
Returns are allowed within 30 days.

Question: What is the return window?

Answer:

If the sources conflict, say so explicitly and cite both [1] and [2].



## Common Mistakes

- Dumping all chunks into the prompt without structure.
- Not citing sources (user cannot verify claims).
- Exceeding the LLM's context window.
- Not telling the model to abstain when context is insufficient.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| LLM ignores context | No grounding instructions | Add explicit grounding |
| Cites wrong source | Labels unclear | Improve source labeling |
| Token limit exceeded | Too many / long chunks | Reduce k or truncate |
| Model makes things up | Not told to abstain | Add "say I don't know" |

## Best Practices

- Always label sources with consistent numbering.
- Include explicit grounding instructions.
- Set a token budget and manage context size.
- Handle contradictions explicitly.
- Test edge cases (no relevant docs, contradictory docs).

## Hands-On Practice

1. **Basic:** Format 3 chunks with source labels.
2. **Guided:** Build a RAG prompt with grounding and test with an LLM.
3. **Independent:** Implement token-budget management.
4. **Realistic:** Test with contradictory sources.
5. **Challenge:** Design context construction for 1, 10, and 100 chunks.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
